# DRIFT MTL — BLR Validation Pass (Colab A100)

Trains and evaluates the EXISTING `backend/mtl_backbone.py` architecture on the real,
verified BLR (VOBL/Bengaluru) multi-task dataset. **This does not produce a pan-India
model.** Training domain is VOBL/Bengaluru only. See `REPORT.md` in this package for
the full audit this notebook is part of.

Steps: install deps -> verify GPU -> build dataset -> train -> evaluate -> compare to
the production XGBoost baseline (already computed, see `evaluation/production_baseline_metrics.json`).

In [ ]:
# 1. Upload or clone the repo so this notebook sits at <repo_root>/colab/
# If you uploaded a zip, unzip it first, then:
import os
assert os.path.exists("../backend/mtl_backbone.py"), "Run this notebook from inside the repo's colab/ folder with the full repo present."
print("Repo layout OK")

In [ ]:
# 2. Install exact dependencies
!pip install -q -r requirements_colab.txt

In [ ]:
# 3. Verify GPU availability and print GPU name
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime > Change runtime type > A100 GPU, then re-run this cell.")

In [ ]:
# 4. Build the BLR training dataset (real CSV, chronological split, class weights, leakage checks)
!python3 build_blr_dataset.py

In [ ]:
# 5. Train the MTL architecture (baseline + class-weighted experiments)
!python3 train_mtl_blr.py

In [ ]:
# 6. Evaluate the selected best checkpoint on the untouched held-out test set
!python3 evaluate_mtl_blr.py

## 7. Compare to production XGBoost baseline

`evaluation/production_baseline_metrics.json` was already computed (in the Claude
sandbox, on CPU, against the real production model artifacts) for the same
VOBL/Bengaluru population and the same 2024-2025 held-out window. Load both files
below to build the side-by-side comparison table required by REPORT.md section 9.
No ranking or "winner" is declared automatically -- read the numbers yourself.

In [ ]:
import json
mtl = json.load(open("evaluation/mtl_test_metrics.json"))
prod = json.load(open("evaluation/production_baseline_metrics.json"))

for hz in ["TS", "CB", "FF"]:
    print(f"=== {hz} ===")
    print("Production XGBoost:", json.dumps(prod["by_hazard"].get(hz, {}), indent=2))
    print("MTL candidate:     ", json.dumps(mtl["by_hazard"].get(hz, {}), indent=2))
    print()

## Collect artifacts

Download the `colab/training/runs/` (checkpoints + curves) and `colab/evaluation/`
(metrics) folders from this Colab session -- these are the reproducible outputs to
bring back for review. Nothing here has been connected to production; see
REPORT.md section 14 for what would be required before any pan-India promotion
could even be considered.